# 블랙아이스 발생 시각화
초음파 거리, 노면 온도, 대기 온도, 조도(LDR), 습도 센서 데이터와 블랙아이스 발생 여부 간의 상관관계를 시각화합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 폰트 설정 초기화 (이전 커널 메모리에 남은 NanumGothic 설정 제거)
plt.rcdefaults()

# 데이터 로드
df = pd.read_csv('measurements.csv')
# 시간 데이터 처리
df['measured_at'] = pd.to_datetime(df['measured_at'])
df['hour'] = df['measured_at'].dt.hour + df['measured_at'].dt.minute / 60.0
# 상태가 'unknown'인 데이터는 시각화의 명확성을 위해 제외
df = df[df['black_ice_status'].isin(['occurred', 'not_occurred'])]
df.head()

In [ ]:
# 분석할 센서 특성들
features = ['temperature', 'road_surface_temp', 'distance_cm_avg', 'ldr_avg', 'humidity']
titles = ['Temperature (C)', 'Road Surface Temp (C)', 'Distance (cm)', 'Illumination (LDR)', 'Humidity (%)']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# 각 변수별로 블랙아이스 발생 여부에 따른 박스플롯(Boxplot) 그리기
for i, (col, title) in enumerate(zip(features, titles)):
    sns.boxplot(x='black_ice_status', y=col, data=df, ax=axes[i], order=['not_occurred', 'occurred'], palette='Set2')
    sns.stripplot(x='black_ice_status', y=col, data=df, ax=axes[i], order=['not_occurred', 'occurred'], color='black', alpha=0.5)
    axes[i].set_title(title)
    axes[i].set_xlabel('Black Ice Status')
    axes[i].set_ylabel('')

# 6번째 빈 그래프 숨기기
axes[5].axis('off')

plt.tight_layout()
plt.show()

위의 박스플롯을 보면 발생 여부를 판가름 짓는 강력한 요인은 **온도와 노면 온도** 임을 알 수 있습니다. 
이 두 가지 핵심 변수를 산점도(Scatter Plot)로 그려서 더 명확히 확인해봅니다.

In [ ]:
# 산점도를 통한 2차원 관계 시각화 (대기 온도 vs 노면 온도)
plt.figure(figsize=(10, 7))

sns.scatterplot(x='temperature', y='road_surface_temp', hue='black_ice_status', 
                style='black_ice_status', s=150, data=df, palette={'not_occurred': 'gray', 'occurred': 'red'})

plt.title('Distribution of Black Ice Occurrence (Air Temp vs Road Temp)', fontsize=14)
plt.xlabel('Air Temperature (C)', fontsize=12)
plt.ylabel('Road Surface Temperature (C)', fontsize=12)

# 0도 기준선 그리기
plt.axvline(x=0, color='blue', linestyle='--', alpha=0.3, label='0C Air Line') 
plt.axhline(y=0, color='blue', linestyle='--', alpha=0.3, label='0C Road Line') 

plt.grid(True, alpha=0.3)
plt.legend(title='Status', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
### 일사량(조도, LDR)과 블랙아이스 발생 여부의 상관관계 심층 분석
기상청 데이터에는 직접적인 '일사량' 데이터가 없으므로, 아두이노에서 측정한 **LDR 조도 센서 값(`ldr_avg`)**을 일사량의 대용 지표로 사용합니다. 
LDR 값이 클수록 빛이 밝고 일사량이 강함을 의미합니다.

조도와 노면 온도의 관계를 파악하여, 아침 햇살(일사량)이 노면을 어떻게 덥히고 블랙아이스에 영향을 주는지 시각화합니다.

In [ ]:
plt.figure(figsize=(12, 6))

# LDR(조도)와 노면 온도(Road Surface Temp)의 상관관계
sns.scatterplot(x='ldr_avg', y='road_surface_temp', hue='black_ice_status', 
                style='black_ice_status', s=150, data=df, palette={'not_occurred': 'gray', 'occurred': 'red'})

plt.title('Illumination (LDR) vs Road Surface Temperature', fontsize=14)
plt.xlabel('Illumination (LDR Value - Proxy for Solar Radiation)', fontsize=12)
plt.ylabel('Road Surface Temperature (C)', fontsize=12)
plt.axhline(y=0, color='blue', linestyle='--', alpha=0.3, label='0C Road Line')
plt.grid(True, alpha=0.3)
plt.legend(title='Status')
plt.tight_layout()
plt.show()

### 분석 결과 해석 (조도/일사량과 블랙아이스)

1. **조도(LDR)와 노면 온도의 양의 상관관계**: LDR 값이 높아질수록(해가 떠서 일사량이 강해질수록) 전반적으로 노면 온도가 서서히 상승하는 경향이 관찰됩니다.
2. **블랙아이스 발생 구역 (Red Dots)**: 블랙아이스가 관측된 시점의 LDR 값은 `300 ~ 750` 사이로 광범위하게 나타납니다. 하지만 흥미롭게도, 일사량(조도)이 강해져서 노면 온도가 1~4℃(영상)로 상승했음에도 불구하고 대기가 영하일 때 블랙아이스가 관측되었습니다.
3. **결론**: 일사량(햇빛)은 노면 온도를 빠르게 올려 기존에 얼어있던 서리나 얼음을 살짝 녹게 만듭니다. 이때 대기 온도가 여전히 영하권(-3도 수준 등)을 유지하고 있다면, **햇빛에 의해 미세하게 녹은 물이 차가운 공기와 만나 다시 투명하게 얼어붙으면서 블랙아이스가 형성**됩니다. 따라서 **'아침 햇살(일사량) 증가 + 차가운 대기 온도'**가 겹치는 시간대가 블랙아이스 발생의 골든타임(위험 시간대)임을 유추할 수 있습니다.